In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver # Every State values is saved on RAM (Once the laptop is shutdown, all the values will be deleted). We use this for DEMO. Not for Production ready.

In [29]:
import os
from dotenv import load_dotenv

In [30]:
load_dotenv()

True

In [31]:
llm = ChatOpenAI(
    model = "openai/gpt-oss-20b",
    api_key = os.getenv("GROK_API_KEY"),
    base_url = "https://api.groq.com/openai/v1",
)

In [32]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explanation : str

In [33]:
def generate_joke(state : JokeState):
    prompt = f"Generate a joke on the topic {state['topic']}"
    response = llm.invoke(prompt).content
    return {
        'joke' : response
    }

In [34]:
def generate_explanation(state : JokeState):
    prompt = f'Write an explanation for the joke - {state['joke']}'
    response = llm.invoke(prompt).content
    return {
        'explanation' : response
    }

In [35]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

In [36]:
checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer = checkpointer) # We are saving every State value with the help of InMemorySaver

* If we pass another topic as "pasta" and with thread_id as "2". Previous thread_id "1" execution would also be saved. As we are using InMemorySaver, if we ShutDown our PC than the history would be gone.

In [37]:
config1 = {
    'configurable' : {"thread_id" : "1"}
}
workflow.invoke(
    {
        "topic" : "pizza"
    },
    config = config1
)

{'topic': 'pizza',
 'joke': 'Why did the pizza start a podcast?  \n\nBecause it wanted to share all its *topping* secrets and *crust* up a good conversation!',
 'explanation': '**Why the joke works – a word‑play deconstruction**\n\n| Element | Literal meaning (pizza) | Punny / figurative meaning | Why it’s funny |\n|---------|-------------------------|----------------------------|----------------|\n| **“topping”** | The cheese, pepperoni, mushrooms, etc. that sit on a pizza. | A play on the phrase “top‑secrets” – i.e., the pizza’s hidden, privileged information. | It turns a mundane pizza ingredient into a sly reference to “secret knowledge.” |\n| **“topping secrets”** | The pizza’s literal secrets about its ingredients. | “Top‑secrets” (important or confidential information). | The double‑reading is a quick, punchy pun that surprises the reader. |\n| **“crust up”** | The edge of a pizza – the hard, chewy part. | Sounds like the idiom “crust up” (to build, prepare, or initiate somethin

In [39]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza start a podcast?  \n\nBecause it wanted to share all its *topping* secrets and *crust* up a good conversation!', 'explanation': '**Why the joke works – a word‑play deconstruction**\n\n| Element | Literal meaning (pizza) | Punny / figurative meaning | Why it’s funny |\n|---------|-------------------------|----------------------------|----------------|\n| **“topping”** | The cheese, pepperoni, mushrooms, etc. that sit on a pizza. | A play on the phrase “top‑secrets” – i.e., the pizza’s hidden, privileged information. | It turns a mundane pizza ingredient into a sly reference to “secret knowledge.” |\n| **“topping secrets”** | The pizza’s literal secrets about its ingredients. | “Top‑secrets” (important or confidential information). | The double‑reading is a quick, punchy pun that surprises the reader. |\n| **“crust up”** | The edge of a pizza – the hard, chewy part. | Sounds like the idiom “crust up” (to build, prepare, o

In [40]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza start a podcast?  \n\nBecause it wanted to share all its *topping* secrets and *crust* up a good conversation!', 'explanation': '**Why the joke works – a word‑play deconstruction**\n\n| Element | Literal meaning (pizza) | Punny / figurative meaning | Why it’s funny |\n|---------|-------------------------|----------------------------|----------------|\n| **“topping”** | The cheese, pepperoni, mushrooms, etc. that sit on a pizza. | A play on the phrase “top‑secrets” – i.e., the pizza’s hidden, privileged information. | It turns a mundane pizza ingredient into a sly reference to “secret knowledge.” |\n| **“topping secrets”** | The pizza’s literal secrets about its ingredients. | “Top‑secrets” (important or confidential information). | The double‑reading is a quick, punchy pun that surprises the reader. |\n| **“crust up”** | The edge of a pizza – the hard, chewy part. | Sounds like the idiom “crust up” (to build, prepare, 